In [53]:
import requests

# 1. Show the raw API response structure (JSON)
url = 'https://power.larc.nasa.gov/api/temporal/daily/point'
params = {
    'parameters': 'T2M,RH2M,PRECTOTCORR,WS10M',
    'community': 'RE',
    'longitude': 74.3587,
    'latitude': 31.5204,
    'start': '20230101',
    'end': '20230107',
    'format': 'JSON'
}
response = requests.get(url, params=params, timeout=30)
data = response.json()

print("Raw JSON keys:", list(data.keys()))
print("\nProperties keys:", list(data['properties'].keys()))
print("\nSample of T2M data:")
print(data['properties']['parameter']['T2M'])

Raw JSON keys: ['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times']

Properties keys: ['parameter']

Sample of T2M data:
{'20230101': 11.95, '20230102': 12.25, '20230103': 11.41, '20230104': 10.18, '20230105': 10.64, '20230106': 10.92, '20230107': 12.08}


In [ ]:
import pandas as pd
import numpy as np

# 2. The daily DataFrame (.head())
params_data = data['properties']['parameter']
df_daily = pd.DataFrame(params_data)

# IMMEDIATELY replace -999.0 with np.nan
df_daily = df_daily.replace(-999.0, np.nan)

df_daily.index = pd.to_datetime(df_daily.index, format='%Y%m%d')
df_daily.index.name = 'date'
df_daily.head()

,T2M,RH2M,PRECTOTCORR,WS10M
date,,,,
2023-01-01,11.95,51.44,0.0,1.64
2023-01-02,12.25,40.31,0.0,1.48
2023-01-03,11.41,39.18,0.0,2.17
2023-01-04,10.18,43.61,0.0,2.52
2023-01-05,10.64,37.98,0.0,2.31


In [6]:
import pandas as pd


def convertToCSV(file, output_file=None, **kwargs):
    df = pd.read_excel(file, **kwargs)
    
    if output_file is None:
        output_file = file.rsplit('.', 1)[0] + '.csv'
    
    df.to_csv(
        output_file,
        index=False,
        encoding='utf-8-sig'
    )
    
    print(f"Converted: {file} → {output_file}")
    return output_file




In [7]:
import pandas as pd
import numpy as np

OFFSEASON_PATTERNS = [
    [0.22, 0.28, 0.28, 0.22],
    [0.24, 0.26, 0.26, 0.24],
    [0.20, 0.30, 0.30, 0.20],
    [0.23, 0.27, 0.27, 0.23],
]
OUTBREAK_BASE = [0.15, 0.35, 0.35, 0.15]

Month_nums = {
    "January":1,"February":2,"March":3,"April":4,
    "May":5,"June":6,"July":7,"August":8,
    "September":9,"October":10,"November":11,"December":12
}
MONTH_TO_START_WEEK = {
    1:1,  2:5,  3:9,  4:14,
    5:18, 6:22, 7:27, 8:31,
    9:36, 10:40, 11:44, 12:48
}

# load full dataset then filter to Punjab only
df_all = pd.read_csv("../data/raw/dengue_data.csv")
df = df_all[df_all["Region"] == "Punjab"].copy().reset_index(drop=True)
print(f"All provinces: {len(df_all)} rows")
print(f"Punjab only:   {len(df)} rows  (expected 60 = 5 years × 12 months)")
print(f"\nPunjab annual totals:")
print(df.groupby("Year")["Dengue_Cases"].sum())

All provinces: 420 rows
Punjab only:   60 rows  (expected 60 = 5 years × 12 months)

Punjab annual totals:
Year
2016     5925
2017     6630
2018     7460
2019    30970
2020     5880
Name: Dengue_Cases, dtype: int64


In [8]:
# Punjab district risk weights
# population: PBS 2017 census
# dengue_history: 0-1 scale based on known outbreak severity
# flood_risk: 0-1 scale based on proximity to Ravi river and flood plains

punjab_districts = pd.DataFrame([
    ["Lahore",          11126285, 0.95, 0.60],
    ["Faisalabad",       7873910, 0.80, 0.40],
    ["Rawalpindi",       5405633, 0.70, 0.50],
    ["Gujranwala",       5014978, 0.75, 0.60],
    ["Multan",           4745209, 0.60, 0.50],
    ["Sheikhupura",      3321029, 0.70, 0.75],
    ["Kasur",            2981000, 0.65, 0.80],
    ["Sialkot",          3893000, 0.60, 0.55],
    ["Sargodha",         3385000, 0.50, 0.45],
    ["Bahawalpur",       3668000, 0.35, 0.40],
    ["Jhang",            2834000, 0.55, 0.70],
    ["Gujrat",           2446000, 0.55, 0.50],
    ["Rahim Yar Khan",   4814000, 0.30, 0.45],
    ["Sahiwal",          2207000, 0.45, 0.40],
    ["Okara",            2797000, 0.40, 0.45],
    ["Narowal",          1453000, 0.50, 0.55],
    ["Hafizabad",        1136000, 0.50, 0.60],
    ["Mandi Bahauddin",  1192000, 0.45, 0.55],
    ["Pakpattan",        1529000, 0.35, 0.40],
    ["Chiniot",          1264000, 0.50, 0.50],
    ["Khushab",          1132000, 0.35, 0.35],
    ["Bhakkar",          1491000, 0.30, 0.35],
    ["Layyah",           1722000, 0.30, 0.40],
    ["Muzaffargarh",     3708000, 0.45, 0.65],
    ["Lodhran",          1715000, 0.35, 0.40],
    ["Vehari",           2895000, 0.40, 0.40],
    ["Khanewal",         2765000, 0.40, 0.45],
    ["Toba Tek Singh",   2191000, 0.45, 0.50],
    ["Nankana Sahib",    1461000, 0.55, 0.55],
    ["Attock",           1575000, 0.40, 0.45],
    ["Chakwal",          1083000, 0.35, 0.30],
    ["Jhelum",           1103000, 0.40, 0.50],
    ["Mianwali",         1560000, 0.30, 0.40],
    ["D.G. Khan",        2982000, 0.35, 0.55],
    ["Rajanpur",         1994000, 0.30, 0.60],
    ["Bahawalnagar",     2978000, 0.30, 0.35],
], columns=["district", "population", "dengue_history", "flood_risk"])

# compute combined risk weight per district
punjab_districts["pop_w"]   = punjab_districts["population"] / punjab_districts["population"].sum()
punjab_districts["hist_w"]  = punjab_districts["dengue_history"] / punjab_districts["dengue_history"].sum()
punjab_districts["flood_w"] = punjab_districts["flood_risk"] / punjab_districts["flood_risk"].sum()

punjab_districts["risk_weight"] = (
    punjab_districts["pop_w"]   * 0.50 +
    punjab_districts["hist_w"]  * 0.30 +
    punjab_districts["flood_w"] * 0.20
)
punjab_districts["risk_weight"] = (
    punjab_districts["risk_weight"] / punjab_districts["risk_weight"].sum()
)

print(f"Districts: {len(punjab_districts)}")
print(f"Weight sum: {punjab_districts['risk_weight'].sum():.6f}  (should be 1.0)")
print("\nTop 5 by risk weight:")
print(punjab_districts[["district","risk_weight"]]
      .sort_values("risk_weight", ascending=False)
      .head())

Districts: 36
Weight sum: 1.000000  (should be 1.0)

Top 5 by risk weight:
     district  risk_weight
0      Lahore     0.076150
1  Faisalabad     0.055855
3  Gujranwala     0.043662
2  Rawalpindi     0.043516
4      Multan     0.038630


In [9]:
def allocate_exact(total, weights):
    """
    Distribute `total` across bins according to `weights`.
    Returns integer list that sums exactly to `total`.
    """
    raw = [total * w for w in weights]
    allocated = [int(x) for x in raw]
    remainder = total - sum(allocated)
    
    # Distribute remainder to largest fractional parts first (fairer)
    fractions = [(i, raw[i] - allocated[i]) for i in range(len(raw))]
    fractions.sort(key=lambda x: x[1], reverse=True)
    
    for i in range(remainder):
        idx = fractions[i % len(fractions)][0]
        allocated[idx] += 1
    
    return allocated

In [10]:
district_rows = []

for _, month_row in df.iterrows():
    month        = str(month_row["Month"])
    month_num    = Month_nums[month]
    year         = int(month_row["Year"])
    total_cases  = int(month_row["Dengue_Cases"])
    total_deaths = int(month_row["Dengue_Deaths"])
    start_week   = MONTH_TO_START_WEEK[month_num]

    for _, dist_row in punjab_districts.iterrows():
        district    = dist_row["district"]
        dist_weight = dist_row["risk_weight"]

        # district's share of Punjab monthly total
        dist_monthly_cases  = round(total_cases  * dist_weight)
        dist_monthly_deaths = round(total_deaths * dist_weight)

        # select temporal weights for this district+month
        if month_num in [7, 8, 9, 10]:
            rng = np.random.default_rng(
                hash(f"{year}{month_num}{district}") % (2**32)
            )
            noise   = rng.normal(0, 0.02, 4)
            weights = [max(0.05, w + n) for w, n in zip(OUTBREAK_BASE, noise)]
        else:
            pattern_idx = (year + month_num + hash(district)) % len(OFFSEASON_PATTERNS)
            weights     = list(OFFSEASON_PATTERNS[pattern_idx])

        weights = [w / sum(weights) for w in weights]

        week_cases  = allocate_exact(dist_monthly_cases,  weights)
        week_deaths = allocate_exact(dist_monthly_deaths, weights)

        for w in range(4):
            epi_week = min(start_week + w, 52)
            district_rows.append({
                "year":      year,
                "month":     month,
                "month_num": month_num,
                "epi_week":  epi_week,
                "district":  district,
                "cases":     week_cases[w],
                "deaths":    week_deaths[w],
            })

df_district_weekly = pd.DataFrame(district_rows)
print(f"Rows generated: {len(df_district_weekly)}")
print(f"Expected:       {len(df) * len(punjab_districts) * 4}  (60 months × 36 districts × 4 weeks)")

Rows generated: 8640
Expected:       8640  (60 months × 36 districts × 4 weeks)


In [11]:
# verify Punjab totals are preserved after disaggregation
original_punjab_total = df["Dengue_Cases"].sum()
district_total        = df_district_weekly["cases"].sum()
diff = abs(original_punjab_total - district_total)

print(f"Original Punjab total cases : {original_punjab_total}")
print(f"District weekly sum         : {district_total}")
print(f"Difference (rounding)       : {diff}  (acceptable if < 200)")

# per-year check
print("\nPer-year totals check:")
orig_by_year = df.groupby("Year")["Dengue_Cases"].sum()
dist_by_year = df_district_weekly.groupby("year")["cases"].sum()
for year in orig_by_year.index:
    o = orig_by_year[year]
    d = dist_by_year.get(year, 0)
    status = "✓" if abs(o-d) < 50 else "WARNING"
    print(f"  {year}: original={o}  districts={d}  {status}")

print(f"\nSample — Lahore 2019 peak:")
print(df_district_weekly[
    (df_district_weekly["district"] == "Lahore") &
    (df_district_weekly["year"] == 2019) &
    (df_district_weekly["epi_week"].between(31, 40))
].to_string())

# save
df_district_weekly.to_csv("../data/raw/punjab_district_weekly.csv", index=False)
print("\nSaved ../data/raw/punjab_district_weekly.csv")

Original Punjab total cases : 56865
District weekly sum         : 56871
Difference (rounding)       : 6  (acceptable if < 200)

Per-year totals check:
  2016: original=5925  districts=5916  ✓
  2017: original=6630  districts=6633  ✓
  2018: original=7460  districts=7463  ✓
  2019: original=30970  districts=30983  ✓
  2020: original=5880  districts=5876  ✓

Sample — Lahore 2019 peak:
      year      month  month_num  epi_week district  cases  deaths
6192  2019     August          8        31   Lahore    138       1
6193  2019     August          8        32   Lahore    362       1
6194  2019     August          8        33   Lahore    319       1
6195  2019     August          8        34   Lahore    129       1
6336  2019  September          9        36   Lahore    105       0
6337  2019  September          9        37   Lahore    233       1
6338  2019  September          9        38   Lahore    210       1
6339  2019  September          9        39   Lahore    116       0
6480  2019 

In [12]:
from backend.utils.nasa_fetcher import fetch_weather
import pandas as pd

district_coords = {
    "Lahore":         (31.5204, 74.3587),
    "Faisalabad":     (31.4504, 73.1350),
    "Rawalpindi":     (33.5651, 73.0169),
    "Gujranwala":     (32.1877, 74.1945),
    "Multan":         (30.1575, 71.5249),
    "Sheikhupura":    (31.7167, 73.9850),
    "Kasur":          (31.1167, 74.4500),
    "Sialkot":        (32.4945, 74.5229),
    "Sargodha":       (32.0836, 72.6711),
    "Bahawalpur":     (29.3956, 71.6836),
    "Jhang":          (31.2681, 72.3181),
    "Gujrat":         (32.5736, 74.0790),
    "Rahim Yar Khan": (28.4202, 70.2952),
    "Sahiwal":        (30.6706, 73.1064),
    "Okara":          (30.8138, 73.4534),
    "Narowal":        (32.1000, 74.8700),
    "Hafizabad":      (32.0714, 73.6883),
    "Mandi Bahauddin":(32.5864, 73.4917),
    "Pakpattan":      (30.3436, 73.3869),
    "Chiniot":        (31.7200, 72.9800),
    "Khushab":        (32.2969, 72.3527),
    "Bhakkar":        (31.6278, 71.0642),
    "Layyah":         (30.9614, 70.9394),
    "Muzaffargarh":   (30.0736, 71.1936),
    "Lodhran":        (29.5333, 71.6333),
    "Vehari":         (30.0453, 72.3519),
    "Khanewal":       (30.3017, 71.9322),
    "Toba Tek Singh": (30.9700, 72.4800),
    "Nankana Sahib":  (31.4500, 73.7100),
    "Attock":         (33.7664, 72.3601),
    "Chakwal":        (32.9328, 72.8571),
    "Jhelum":         (32.9350, 73.7203),
    "Mianwali":       (32.5836, 71.5433),
    "D.G. Khan":      (30.0500, 70.6333),
    "Rajanpur":       (29.1042, 70.3300),
    "Bahawalnagar":   (29.9978, 73.2536),
}

weather_frames = []
for district, (lat, lon) in district_coords.items():
    print(f"Fetching {district}...")
    try:
        df_wx = fetch_weather(lat, lon, "20160101", "20201231")
        df_wx["district"] = district
        weather_frames.append(df_wx)
    except Exception as e:
        print(f"  FAILED {district}: {e}")

df_weather = pd.concat(weather_frames, ignore_index=True)

df_district = pd.read_csv("../data/raw/punjab_district_weekly.csv")
df_merged = pd.merge(
    df_district, df_weather,
    on=["district", "year", "epi_week"],
    how="inner"
)

print(f"Merged shape: {df_merged.shape}")
df_merged.to_csv("../data/processed/punjab_district_weekly_merged.csv", index=False)
print("Saved punjab_district_weekly_merged.csv")

Fetching Lahore...
Fetching Faisalabad...
Fetching Rawalpindi...
Fetching Gujranwala...
Fetching Multan...
Fetching Sheikhupura...
Fetching Kasur...
Fetching Sialkot...
Fetching Sargodha...
Fetching Bahawalpur...
Fetching Jhang...
Fetching Gujrat...
Fetching Rahim Yar Khan...
Fetching Sahiwal...
Fetching Okara...
Fetching Narowal...
Fetching Hafizabad...
Fetching Mandi Bahauddin...
Fetching Pakpattan...
Fetching Chiniot...
Fetching Khushab...
Fetching Bhakkar...
Fetching Layyah...
Fetching Muzaffargarh...
Fetching Lodhran...
Fetching Vehari...
Fetching Khanewal...
Fetching Toba Tek Singh...
Fetching Nankana Sahib...
Fetching Attock...
Fetching Chakwal...
Fetching Jhelum...
Fetching Mianwali...
Fetching D.G. Khan...
Fetching Rajanpur...
Fetching Bahawalnagar...
Merged shape: (8640, 11)
Saved punjab_district_weekly_merged.csv
